# 10.7 Overfitting vs Underfitting

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.07_overfitting-vs-underfitting.ipynb)

Companion code for Section 10.7 of the book. Run the cells in order from top to bottom.

**Example 10.7.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import learning_curve, validation_curve, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Plot learning curves for 3 models ────────────────────────────
models = {
    'Underfitting\n(Logistic Regression)':   Pipeline([('sc',StandardScaler()),
        ('lr',LogisticRegression(C=0.0001, max_iter=1000))]),
    'Good Fit\n(Random Forest)':             RandomForestClassifier(n_estimators=100, random_state=42),
    'Overfitting\n(Decision Tree depth=None)': DecisionTreeClassifier(max_depth=None, random_state=42),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Overfitting vs Underfitting - Learning Curves', fontsize=14, fontweight='bold')

for ax, (name, model) in zip(axes, models.items()):
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y,
        train_sizes=np.linspace(0.1, 1.0, 12),
        cv=cv, scoring='accuracy', n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    test_mean  = test_scores.mean(axis=1)
    test_std   = test_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, 'b-o', ms=6, lw=2, label='Train')
    ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.1, color='blue')
    ax.plot(train_sizes, test_mean, 'r-o', ms=6, lw=2, label='CV')
    ax.fill_between(train_sizes, test_mean-test_std, test_mean+test_std, alpha=0.1, color='red')

    # Annotate gap
    gap = train_mean[-1] - test_mean[-1]
    ax.annotate(f'Gap={gap:.3f}',
               xy=(train_sizes[-1], (train_mean[-1]+test_mean[-1])/2),
               xytext=(-80, 0), textcoords='offset points',
               fontsize=10, color='purple', fontweight='bold',
               arrowprops=dict(arrowstyle='->', color='purple'))

    ax.set_xlabel('Training Set Size'); ax.set_ylabel('Accuracy')
    ax.set_title(name, fontsize=11); ax.legend(fontsize=9)
    ax.set_ylim(0.5, 1.05); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Validation curves: hyperparameter effect on bias-variance ─────
print('=== Validation Curves: How max_depth Affects Bias-Variance ===')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Validation Curves - Bias-Variance Tradeoff', fontsize=14, fontweight='bold')

# Decision Tree: max_depth
ax = axes[0]
depths = [1, 2, 3, 4, 5, 7, 10, 15, None]
train_acc, test_acc = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    tr = cross_val_score(dt, X, y, cv=cv, scoring='accuracy').mean()
    te_scores = cross_val_score(dt, X, y, cv=cv, scoring='accuracy')
    train_dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    train_dt.fit(X, y)
    train_acc.append(train_dt.score(X, y))
    test_acc.append(te_scores.mean())

x_vals = range(len(depths))
ax.plot(x_vals, train_acc, 'b-o', ms=7, lw=2.5, label='Train')
ax.plot(x_vals, test_acc,  'r-o', ms=7, lw=2.5, label='CV')
ax.axvline(x=3, color='green', ls='--', lw=2, label='Optimal depth=4')
ax.set_xticks(x_vals)
ax.set_xticklabels([str(d) for d in depths])
ax.set_xlabel('max_depth'); ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: max_depth\n(left=underfit, right=overfit)')
ax.legend(); ax.grid(alpha=0.3)

# Logistic Regression: C (regularisation strength)
ax = axes[1]
C_values = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
from sklearn.model_selection import validation_curve
pipe_lr = Pipeline([('sc',StandardScaler()),('lr',LogisticRegression(max_iter=2000))])
train_s, test_s = validation_curve(
    pipe_lr, X, y, param_name='lr__C', param_range=C_values,
    cv=cv, scoring='accuracy', n_jobs=-1
)
ax.semilogx(C_values, train_s.mean(axis=1), 'b-o', ms=7, lw=2.5, label='Train')
ax.semilogx(C_values, test_s.mean(axis=1),  'r-o', ms=7, lw=2.5, label='CV')
ax.fill_between(C_values, test_s.mean(axis=1)-test_s.std(axis=1),
               test_s.mean(axis=1)+test_s.std(axis=1), alpha=0.15, color='red')
ax.set_xlabel('C (regularisation strength)'); ax.set_ylabel('Accuracy')
ax.set_title('Logistic Regression: C value\n(left=underfit, right=overfit)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('validation_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nDiagnostic Guide:')
print('  Train ≈ CV (both low):   UNDERFITTING  → increase complexity')
print('  Train >> CV (large gap): OVERFITTING   → add regularisation, get more data')
print('  Train ≈ CV (both high):  GOOD FIT      → deploy')